In [5]:
import os
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from glob import glob
import numpy as np
from spectral.io import envi
from tqdm import tqdm

os.chdir('/store/carroll/sbgplants/')

In [2]:
# file paths
raw = 'data/raw'
rdn_fol = '/store/carroll/col/data/2018/raw/L1/'
out_folder = 'data/out_csv'

table = 'extracted_observations'

In [3]:
# load relevant output tables

pixel = pd.read_csv(os.path.join(out_folder, 'pixel.csv'))
fids = pixel.granule_id.unique()

In [12]:
# extract obs per px

fps = [x for x in glob(os.path.join(rdn_fol, '*/*obs_ort.hdr')) if any(xx in x for xx in fids)]
id_cols = pixel.columns
val_cols = envi.read_envi_header(fps[0])['band names']

out = []

for fp in tqdm(fps):
    fid = fp.split('/')[-1].removesuffix('_rdn_obs_ort.hdr')
    tmp = pixel[pixel['granule_id']==fid].copy()
    # extract obs
    obs = envi.open(fp).open_memmap()
    r = tmp['glt_row']; c=tmp['glt_column']
    vals = obs[r, c, :]
    # format df
    tmp = pd.concat([tmp, pd.DataFrame(vals, index=tmp.index, columns=val_cols)], axis=1)
    tmp = tmp.melt(id_vars=id_cols, value_vars=val_cols, var_name='obs_type', value_name='obs_value')
    out.append(tmp)

df = pd.concat(out)
df = df[df['obs_type']!='ATCOR to-sensor zenith']

100%|██████████████████████████████████████████████████████████████████████████████████████████| 61/61 [00:03<00:00, 17.21it/s]


In [27]:
# prepare & populate out table
out_table = df.copy()

# map obs_type to enumerated values
out_table = out_table[out_table.obs_type!='ATCOR to-sensor zenith']
obs_type = {
    'path length' : 'path length',
    'to-sensor azimuth' : 'to-sensor-azimuth',
    'to-sensor zenith' : 'to-sensor-zenith',
    'to-sun azimuth' : 'to-sun-azimuth',
    'to-sun zenith' : 'to-sun-zenith',
    'phase' : 'solar phase',
    'slope' : 'slope',
    'aspect' : 'aspect',
    'cosine i' : 'cosine i',
    'gps time' : 'UTC time'
}
out_table.loc[:,'obs_type'] = out_table['obs_type'].map(obs_type)

out_table['obs_id'] = range(len(out_table))
out_table = out_table[['obs_id','pixel_id','obs_type','obs_value']]

out_table

,obs_id,pixel_id,obs_type,obs_value
0,0,1303,path length,1683.907959
1,1,1304,path length,1684.000244
2,2,1305,path length,1683.652344
3,3,1306,path length,1683.788208
4,4,1303,to-sensor-azimuth,182.968033
...,...,...,...,...
2425,152355,3799,UTC time,17.218855
2426,152356,3800,UTC time,17.218859
2427,152357,3801,UTC time,17.218853
2428,152358,3802,UTC time,17.218855


In [28]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)